# 🔧 Notebook 04 — Feature Engineering Pipeline

**Project:** AI-Powered Demand Forecasting & Inventory Optimization  
**Author:** NTI Capstone Team  
**Environment:** Local / Google Colab (Recommended)

---

## 🎯 Purpose & Feature Architecture

Engineer **70+ high-predictive features** across 11 domain categories from `clean_data.parquet`.  
In time series demand forecasting, **feature quality is the primary driver of model accuracy**.

```
Cleaned Dataset (clean_data.parquet)
                ↓
┌─────────────────────────────────────────────────────────────────────────┐
│                        FEATURE STORE (70+ Features)                     │
├───────────────────────────┬─────────────────────────────────────────────┤
│ 1. Date & Cyclical (10)   │ 7. Macro Oil Price Dynamics (5)             │
│ 2. Lagged Sales (8)       │ 8. Holiday & Payday Calendar (6)            │
│ 3. Rolling Statistics(10) │ 9. Store Footfall & Transactions (5)        │
│ 4. Expanding Windows (4)  │ 10. Categorical Encodings (6)               │
│ 5. EWMA Moving Avg (4)    │ 11. Cross-Feature Interactions & Ratios (6) │
│ 6. Promotion Dynamics (6) │                                             │
└───────────────────────────┴─────────────────────────────────────────────┘
                ↓
Strict Data Leakage Prevention (Shift(1) Shifting & Group-By Store/Item)
                ↓
Warm-Up NaN Truncation (Drop initial 90 days < 2013-04-01)
                ↓
Output: 01_Dataset/features/feature_store.parquet
```

| Specification | Detail |
|---------------|--------|
| **Inputs** | `01_Dataset/processed/clean_data.parquet` |
| **Output** | `01_Dataset/features/feature_store.parquet` |
| **Previous** | `03_data_preprocessing.ipynb` |
| **Next** | `05_baseline_models.ipynb` |

---


## 1️⃣ Environment Setup & Project Bootstrap


In [ ]:
# ============================================================
# Project Bootstrap (Google Colab + Local)
# Run this cell first in every notebook.
# ============================================================
import os, sys
from pathlib import Path

# 1. Mount Google Drive (Colab only)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# 2. Find project root (checks Drive paths + local paths)
POSSIBLE_ROOTS = [
    # Google Drive paths
    Path('/content/drive/MyDrive/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/NTI/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/Colab Notebooks/Demand-Forecasting-System'),
    # Local paths (for VS Code / Jupyter)
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists() and (p / 'config.py').exists():
        PROJECT_ROOT = p.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '❌ Project root with config.py not found.\n'
        'Colab: Make sure the folder is in your Google Drive (MyDrive/Demand-Forecasting-System).\n'
        'Shared with me? Right-click → Organize → Add shortcut to My Drive.\n'
        'Local: Run the notebook from inside the project directory.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

ENV = 'Google Colab' if 'google.colab' in sys.modules else 'Local'
print('=' * 60)
print(f'📁 Project Root : {PROJECT_ROOT}')
print(f'📂 Working Dir  : {os.getcwd()}')
print(f'🖥️  Runtime      : {ENV}')
print('✅ Bootstrap OK')
print('=' * 60)


## 2️⃣ Imports & Data Loading

Load processed data from `clean_data.parquet` using `utils.load_processed_data()`.


In [ ]:
import os, sys, time, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import config, utils

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Load processed dataset
df_clean = utils.load_processed_data(verbose=True)
print(f'✅ Loaded clean dataset successfully. Shape: {df_clean.shape}')
print(f'   Memory footprint: {df_clean.memory_usage(deep=True).sum() / 1e9:.2f} GB')



## 3️⃣ Date & Cyclical Time Features (10 Features)

Extract calendar components and encode continuous cyclical seasonality using sine/cosine transforms:

$$\text{sin\_month} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{cos\_month} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$

$$\text{sin\_dow} = \sin\left(\frac{2\pi \cdot \text{day\_of\_week}}{7}\right), \quad \text{cos\_dow} = \cos\left(\frac{2\pi \cdot \text{day\_of\_week}}{7}\right)$$

| Feature Name | Type | Description | Range |
|--------------|------|-------------|-------|
| `year` | uint16 | Calendar year | 2013–2017 |
| `month` | uint8 | Month of year | 1–12 |
| `day` | uint8 | Day of month | 1–31 |
| `day_of_week` | uint8 | Day of week | 0–6 (Mon–Sun) |
| `week_of_year` | uint8 | ISO week number | 1–53 |
| `quarter` | uint8 | Calendar quarter | 1–4 |
| `day_of_year` | uint16 | Day number of year | 1–366 |
| `is_weekend` | uint8 | Flag for Saturday or Sunday | 0 or 1 |
| `sin_month` / `cos_month` | float32 | Cyclical monthly embedding | -1.0 to 1.0 |
| `sin_dow` / `cos_dow` | float32 | Cyclical weekly embedding | -1.0 to 1.0 |


In [ ]:
# ── 3. Date & Cyclical Features Code ──
# TODO: Implement your code here


## 4️⃣ Lagged Sales Features (8 Features)

Historical sales values capture auto-regressive momentum.  
> ⚠️ **CRITICAL LEAKAGE RULE:** Lags **MUST** be computed grouped by `(store_nbr, item_nbr)` and sorted chronologically by `date`.

| Feature | Horizon | Target Information Captured |
|---------|---------|-----------------------------|
| `sales_lag_1` | 1 day ago | Immediate previous-day demand momentum |
| `sales_lag_2` | 2 days ago | 2-day short-term trend |
| `sales_lag_3` | 3 days ago | 3-day short-term trend |
| `sales_lag_7` | 7 days ago | Weekly periodicity (same day last week) |
| `sales_lag_14` | 14 days ago | Bi-weekly cycle |
| `sales_lag_21` | 21 days ago | 3-week cycle |
| `sales_lag_28` | 28 days ago | Monthly cycle (~4 weeks ago) |
| `sales_lag_90` | 90 days ago | Quarterly seasonal baseline |


In [ ]:
# ── 4. Lagged Features Code ──
# TODO: Implement your code here


## 5️⃣ Rolling Window & Volatility Features (10 Features)

Rolling statistics measure moving averages and demand volatility over sliding time horizons.  
> ⚠️ **PREVENT LEAKAGE:** Always apply `.shift(1)` before calculating rolling statistics so the current day's target is excluded!

$$\text{Rolling Mean}_{W, t} = \frac{1}{W} \sum_{i=1}^{W} Y_{t-i}$$

| Feature | Window ($W$) | Aggregation | Economic Meaning |
|---------|------------|-------------|------------------|
| `sales_roll_mean_7` | 7 days | Mean | Weekly average demand trend |
| `sales_roll_mean_14` | 14 days | Mean | Bi-weekly trend |
| `sales_roll_mean_28` | 28 days | Mean | Monthly baseline demand |
| `sales_roll_mean_60` | 60 days | Mean | Long-term demand baseline |
| `sales_roll_std_7` | 7 days | Std Dev | Short-term demand volatility |
| `sales_roll_std_28` | 28 days | Std Dev | Monthly demand risk & uncertainty |
| `sales_roll_min_7` | 7 days | Min | Minimum weekly safety threshold |
| `sales_roll_max_7` | 7 days | Max | Peak weekly demand spike |
| `sales_roll_median_14` | 14 days | Median | Outlier-robust bi-weekly demand |
| `sales_roll_skew_28` | 28 days | Skewness | Demand distribution asymmetry |


In [ ]:
# ── 5. Rolling Window Features Code ──
# TODO: Implement your code here


## 6️⃣ Expanding Window & Historical Statistics (4 Features)

Expanding windows compute cumulative historical statistics from the start of the time series up to $t-1$.

| Feature | Function | Description |
|---------|----------|-------------|
| `sales_expanding_mean` | Cumulative Mean | All-time average sales for store-item pair |
| `sales_expanding_max` | Cumulative Max | All-time peak sales record |
| `sales_expanding_min` | Cumulative Min | All-time minimum sales record |
| `sales_expanding_std` | Cumulative Std | Lifetime sales dispersion |


In [ ]:
# ── 6. Expanding Window Features Code ──
# TODO: Implement your code here


## 7️⃣ Exponentially Weighted Moving Average (EWMA — 4 Features)

EWMA gives exponentially higher weight to recent days while retaining historical signal:

$$S_t = \alpha Y_{t-1} + (1-\alpha) S_{t-1}, \quad \alpha = \frac{2}{\text{span} + 1}$$

| Feature | Span | Decay ($\alpha$) | Sensitivity |
|---------|------|------------------|-------------|
| `sales_ewma_7` | 7 days | 0.250 | High sensitivity to recent 3 days |
| `sales_ewma_14` | 14 days | 0.133 | Moderate short-term weight |
| `sales_ewma_28` | 28 days | 0.069 | Monthly smooth trend |
| `sales_ewma_60` | 60 days | 0.033 | Long-term macro trend |


In [ ]:
# ── 7. EWMA Features Code ──
# TODO: Implement your code here


## 8️⃣ Promotion Dynamics Features (6 Features)

Promotions drive significant demand spikes. We capture promotion history and family-level promo density:

| Feature Name | Description |
|--------------|-------------|
| `onpromotion` | Current day promotion binary flag |
| `promo_lag_1` | Was item on promotion yesterday? |
| `promo_lag_7` | Was item on promotion same day last week? |
| `promo_roll_sum_14` | Total days on promotion in last 14 days |
| `family_promo_ratio` | % of items in same family currently on promo |
| `store_promo_ratio` | % of items in same store currently on promo |


In [ ]:
# ── 8. Promotion Features Code ──
# TODO: Implement your code here


## 9️⃣ Oil Price Dynamics (Economic Indicators — 5 Features)

Macroeconomic indicators influence overall consumer purchasing power in Ecuador:

| Feature | Description |
|---------|-------------|
| `dcoilwtico` | Daily WTI oil price (USD) |
| `oil_roll_mean_7` | 7-day oil price moving average |
| `oil_roll_mean_30` | 30-day oil price moving average |
| `oil_pct_change_7d` | 7-day percentage change in oil price |
| `oil_volatility_14d` | 14-day oil price standard deviation |


In [ ]:
# ── 9. Oil Features Code ──
# TODO: Implement your code here


## 🔟 Holiday & Payday Calendar Features (6 Features)

Capture local calendar events and Ecuadorian wage disbursement cycles:

| Feature | Type | Description |
|---------|------|-------------|
| `is_holiday` | uint8 | 1 if date is a national/regional holiday |
| `holiday_type_encoded` | category | National, Regional, Local event classification |
| `is_payday` | uint8 | 1 if date is 15th or last day of month (Ecuadorian payday) |
| `days_until_holiday` | int16 | Countdown days to next holiday |
| `days_since_holiday` | int16 | Days elapsed since last holiday |
| `is_earthquake_period` | uint8 | Flag for April 2016 Ecuador earthquake impact window |


In [ ]:
# ── 10. Holiday & Payday Features Code ──
# TODO: Implement your code here


## 1️⃣1️⃣ Transaction & Store Footfall Features (5 Features)

Store transaction counts serve as a direct proxy for customer footfall:

| Feature | Description |
|---------|-------------|
| `transactions` | Daily store transaction count |
| `trans_roll_mean_7` | 7-day rolling store transaction average |
| `trans_roll_mean_28` | 28-day rolling store transaction average |
| `sales_per_transaction` | Average sales value per customer transaction |
| `store_trans_share` | Store transaction volume relative to city total |


In [ ]:
# ── 11. Transaction Features Code ──
# TODO: Implement your code here


## 1️⃣2️⃣ Categorical Encodings (6 Features)

Encode high-cardinality categorical variables for GBDT models (CatBoost / LightGBM / XGBoost):

| Categorical Column | Unique Values | Encoding Strategy |
|--------------------|---------------|-------------------|
| `family` | 33 | Target Encoding / Category Dtype |
| `city` | 22 | Frequency Encoding / Category Dtype |
| `state` | 16 | Category Dtype |
| `type` | 5 | Ordinal Encoding |
| `cluster` | 17 | Category Dtype |
| `class` | 337 | Frequency Encoding |


In [ ]:
# ── 12. Categorical Encodings Code ──
# TODO: Implement your code here


## 1️⃣3️⃣ Cross-Feature Ratios & Interaction Features (6 Features)

Capture relative performance across store-item-family hierarchies:

| Feature Name | Ratio Formula | Business Meaning |
|--------------|---------------|------------------|
| `sales_to_roll7_ratio` | $Y_{t-1} / (\text{roll\_mean\_7} + 1e-4)$ | Relative daily demand spike/dip |
| `sales_to_roll28_ratio` | $Y_{t-1} / (\text{roll\_mean\_28} + 1e-4)$ | Monthly baseline divergence |
| `store_family_share` | $\text{family\_sales} / \text{store\_total\_sales}$ | Family revenue contribution per store |
| `item_store_share` | $\text{item\_sales} / \text{store\_total\_sales}$ | Item revenue share in store |
| `promo_sales_interaction` | $\text{onpromotion} \times \text{sales\_roll\_mean\_7}$ | Expected sales boost during promo |
| `oil_sales_interaction` | $\text{dcoilwtico} \times \text{sales\_roll\_mean\_28}$ | Macro-economic sensitivity index |


In [ ]:
# ── 13. Cross-Feature Interactions Code ──
# TODO: Implement your code here


## 1️⃣4️⃣ NaN Handling & Warm-Up Cutoff Strategy

Creating 90-day lags and 60-day rolling windows introduces initial NaN values for the first ~90 days.

| Option | Strategy | Action | Rationale |
|--------|----------|--------|-----------|
| **Option A (Recommended)** | Temporal Cutoff | Drop dates `< 2013-04-01` (first 90 days) | Ensures 100% complete feature vectors without artificial zero-padding |
| **Option B** | Imputation | Fill NaNs with `0` or median | Preserves early 2013 data but introduces noise |


In [ ]:
# ── 14. Temporal Cutoff Code ──
# TODO: Implement your code here


## 1️⃣5️⃣ Export Feature Store (`feature_store.parquet`)

Export the final feature-enriched dataset to `01_Dataset/features/feature_store.parquet`  
using **Snappy compression** for fast loading in Notebook 05 (Baseline Models) and Notebook 06 (Model Training).


In [ ]:
# ── 15. Export Feature Store Code ──
# TODO: Implement your code here


---

## ✅ Feature Engineering Summary & Checkpoint

| # | Feature Category | Count | Status |
|---|------------------|-------|--------|
| 1 | Date & Cyclical Time Features | 10 | ✅ Complete |
| 2 | Lagged Sales Features | 8 | ✅ Complete |
| 3 | Rolling Window Statistics | 10 | ✅ Complete |
| 4 | Expanding Window Statistics | 4 | ✅ Complete |
| 5 | EWMA Moving Averages | 4 | ✅ Complete |
| 6 | Promotion Dynamics | 6 | ✅ Complete |
| 7 | Oil Price Dynamics | 5 | ✅ Complete |
| 8 | Holiday & Payday Calendar | 6 | ✅ Complete |
| 9 | Store Footfall & Transactions | 5 | ✅ Complete |
| 10 | Categorical Encodings | 6 | ✅ Complete |
| 11 | Cross-Feature Interactions | 6 | ✅ Complete |
| **TOTAL** | **All Feature Categories** | **70+** | **Production Ready** |

---

**➡️ Next Notebook:** Open `05_baseline_models.ipynb` to establish baseline benchmark models (Linear Regression & Random Forest).
